In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
import os
openai_api_key = OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

In [ ]:
air_quality_report_agent = Agent(
    role="Senior air quality research and report agent",
	goal="Be a senior professional reporter "
        "Draft air quality reports, researching facts",
	backstory=(
		"You work as a senior air quality reporter agent for different cities in the worlds "
        " are now working on providing "
		"research to {city}, a super important activity for the newspaper "
        " for your research."
		"You need to make sure that you provide the best research report with facts and list references correctly!"
		"Make sure to provide full complete detailed research reports, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

In [ ]:
quality_assurance_agent = Agent(
	role="Report Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support report quality assurance in your team",
	backstory=(
		"You work at leading news paper brnad and "
        "are now working with your team "
		"on a request from {city} ensuring that "
        "the air quality research and report agent is "
		"providing the best support possible.\n"
		"You need to make sure that the air quality research and report agent "
        "is providing full"
		"complete answers, and makes no assumptions. Make sure to double check that air quality report agent reports all the necessary sources from where the data is taken and is correct! "
	),
	verbose=True
)

In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://www.aqi.in/in/dashboard/india/maharashtra/pune"
)

Above code will instantiate a document scraper tool.
The tool will scrape a page **(only 1 URL)** of the CrewAI documentation.

In [ ]:
research_task = Task(
    description=(
        "Quickly do a thourough research for the following {city}:\n"

		"Make sure to use everything you know "
        "to provide the best and comprehensive news article as possible."
		"You must strive to provide a complete "
        "and accurate response with all references"
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=air_quality_report_agent,
)

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the air quality report agent for city: {city}. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for a news report.\n"
        "Verify that all parts such as source of the information of the city {city} "
        "have been addressed "
		"thoroughly, with a news reporting tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported from a news perspective and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative news article response "
        "ready to be published as a news report.\n"
		"Please maintain a very formal tone throughout. "
    ),
    agent=quality_assurance_agent,
)


In [ ]:
crew = Crew(
  agents=[air_quality_report_agent, quality_assurance_agent],
  tasks=[research_task, quality_assurance_review],
  verbose=2,
  memory=True
)

In [ ]:
inputs = {
    "city": "Pune"
}
result = crew.kickoff(inputs=inputs)

In [ ]:
from IPython.display import Markdown
Markdown(result)